# GVH–CHH · A1 · Trois compartiments de gaz
**CHH-COMP-01 · v0.1 · Brouillon pré-gel**  
Protocole CHH figé · [Budget d’erreur à deux compartiments](./CHH_COMP_01_Gaz_Budget_Erreur_A1_v01.ipynb)

## Niveau 1 — CHH-STRUCTURE et INVERSE-INFERENCE

Trois compartiments rigides de volumes connus contiennent le même gaz idéal. L’enceinte est fermée à la matière et la température est la même dans les trois compartiments avant ouverture et à l’équilibre final. Les pressions sont absolues (kPa), les volumes en litres. Les inconnues sont les trois pressions initiales. On compare trois ensembles de mesures et, si deux pressions initiales peuvent être mesurées, on cherche quel compartiment laisser à reconstruire.

## Niveau 2 — KNOWN-PHYSICS

La loi du gaz idéal et la conservation de la quantité de matière donnent
$$p_f=w_1p_1+w_2p_2+w_3p_3,\qquad w_i=\frac{V_i}{V_1+V_2+V_3}.$$
La pression finale seule donne une équation de rang 1. Avec $p_1$, le rang passe à 2 et on identifie $w_2p_2+w_3p_3$. Avec $p_1,p_2$, le rang vaut 3 si $w_3>0$. Le déterminant de la matrice des mesures ordonnées $(p_1,p_2,p_f)$ vaut $w_3$. Pour une cible quelconque $j$,
$$\widehat p_j=\frac{p_f^{\rm mes}-\sum_{i\ne j}w_ip_i^{\rm mes}}{w_j}.$$
Avec des bruits de pression indépendants, centrés et de même écart-type $\sigma$,
$$\sigma_{\widehat p_j-p_j}=\frac{\sigma}{w_j}\sqrt{1+\sum_{i\ne j}w_i^2}.$$
À volumes exacts, cette relation est exacte. Comme la variance normalisée vaut $(1+\sum_i w_i^2)/w_j^2-1$, la cible de plus grand volume minimise la RMSE théorique dans ces conditions. À $w_j=0$, la pression finale est insensible à la cible.

## Niveau 3 — NUMERICAL-TEST

Plan fixé dans cette version. Volumes de 6, 3 et 1 L. Générer 20 000 systèmes avec trois pressions indépendantes uniformes entre 80 et 120 kPa. Pour chaque système, utiliser quatre bruits gaussiens indépendants d’écart-type 0,20 kPa, un par pression initiale et un pour la pression finale. Les mêmes systèmes et mêmes erreurs sont utilisés pour comparer les trois placements. Les masses de gaz restent implicites. Les valeurs synthétiques exactes servent à générer les observations et à évaluer les erreurs.

Vérifier par des témoins distincts l’ambiguïté des deux premiers ensembles de mesures. Vérifier les trois rangs, une erreur maximale sans bruit inférieure à $10^{-10}$ kPa, l’accord simulation–théorie à 3 % près pour chaque cible et le classement attendu des trois placements.

In [1]:
from fractions import Fraction
from math import isclose, sqrt
from random import Random
from statistics import fmean

VOLUMES = (6, 3, 1)
WEIGHTS = tuple(Fraction(v, sum(VOLUMES)) for v in VOLUMES)
SIGMA = 0.20
COUNT = 20_000
SEED = 20261122

def exact_rank(rows):
    matrix = [list(map(Fraction, row)) for row in rows]
    rank = 0
    for col in range(3):
        pivot = next((i for i in range(rank, len(matrix)) if matrix[i][col]), None)
        if pivot is None:
            continue
        matrix[rank], matrix[pivot] = matrix[pivot], matrix[rank]
        for i in range(rank + 1, len(matrix)):
            factor = matrix[i][col] / matrix[rank][col]
            matrix[i] = [a - factor*b for a, b in zip(matrix[i], matrix[rank], strict=True)]
        rank += 1
    return rank

def final_pressure(initial, weights):
    return sum(w*p for w, p in zip(weights, initial, strict=True))

def reconstruct(final, observed, weights, target):
    if weights[target] == 0:
        return None
    return (final - sum(weights[i]*p for i, p in observed.items())) / weights[target]

ranks = [exact_rank(rows) for rows in (
    [WEIGHTS], [WEIGHTS, (1, 0, 0)], [WEIGHTS, (1, 0, 0), (0, 1, 0)]
)]
assert ranks == [1, 2, 3]
assert exact_rank([(Fraction(2,3), Fraction(1,3), 0), (1,0,0), (0,1,0)]) == 2
assert reconstruct(100, {0: 100, 1: 100}, (0.6, 0.4, 0), 2) is None
for alternative in ((101, 98, 100), (100, 101, 97)):
    assert final_pressure(alternative, WEIGHTS) == final_pressure((100, 100, 100), WEIGHTS)
print("Rangs :", ranks)
print("Même pression finale : (100, 100, 100), (101, 98, 100), (100, 101, 97).")
print("Le premier et le troisième triplets donnent aussi p1 = 100 kPa.")

Rangs : [1, 2, 3]
Même pression finale : (100, 100, 100), (101, 98, 100), (100, 101, 97).
Le premier et le troisième triplets donnent aussi p1 = 100 kPa.


In [2]:
weights = tuple(map(float, WEIGHTS))
rng = Random(SEED)
draws = [(
    tuple(rng.uniform(80, 120) for _ in range(3)),
    tuple(rng.gauss(0, SIGMA) for _ in range(4))
) for _ in range(COUNT)]

errors = []
print("Cible | volume (L) | RMSE simulée (kPa) | RMSE théorique (kPa) | erreur max sans bruit (kPa)")
for target in range(3):
    measured_indices = tuple(i for i in range(3) if i != target)
    squared = []
    max_exact = 0.0
    for pressures, noise in draws:
        pf = final_pressure(pressures, weights)
        observed = {i: pressures[i] for i in measured_indices}
        exact = reconstruct(pf, observed, weights, target)
        max_exact = max(max_exact, abs(exact - pressures[target]))
        noisy = {i: p + noise[i] for i, p in observed.items()}
        estimate = reconstruct(pf + noise[3], noisy, weights, target)
        squared.append((estimate - pressures[target])**2)
    rmse = sqrt(fmean(squared))
    theory = SIGMA * sqrt(1 + sum(weights[i]**2 for i in measured_indices)) / weights[target]
    assert max_exact < 1e-10
    assert isclose(rmse, theory, rel_tol=0.03)
    errors.append(rmse)
    print(f"{target+1:5} | {VOLUMES[target]:10} | {rmse:18.3f} | {theory:20.3f} | {max_exact:.2e}")

assert errors[0] < errors[1] < errors[2]
print(f"{COUNT:,} systèmes par placement | graine {SEED}")
print("PASS_SYNTHETIC_THREE_GAS_COMPARTMENTS_SENSOR_PLACEMENT")

Cible | volume (L) | RMSE simulée (kPa) | RMSE théorique (kPa) | erreur max sans bruit (kPa)
    1 |          6 |              0.348 |                0.350 | 2.84e-14
    2 |          3 |              0.778 |                0.780 | 5.68e-14
    3 |          1 |              2.398 |                2.408 | 1.56e-13
20,000 systèmes par placement | graine 20261122
PASS_SYNTHETIC_THREE_GAS_COMPARTMENTS_SENSOR_PLACEMENT


## Niveau 4 — Verdict et portée

Dans ce modèle, deux pressions initiales mesurées et une pression finale suffisent à reconstruire la troisième pression initiale. Pour réduire la RMSE, on mesure directement les pressions initiales des deux plus petits compartiments et on reconstruit celle du plus grand. Cette règle est démontrée ici pour des volumes exacts et des bruits de pression indépendants, de même écart-type. Une instrumentation hétérogène ou des volumes incertains exigent de recalculer le budget. Le verdict porte sur un contrôle synthétique statique.